# Parte 0 - Tres fallas que no fallan

**Nombres: Jessica Ballesteros, Miguel Álvarez, Darlyn Ludeña**

## 0.a — El PDF que se indexa vacío (3 %) 

**`ejemplos/instructivo_escaneado.pdf` no tiene capa de texto. Ingesta la carpeta rag_pipeline.py y pega la salida de la ingesta. Después, en `ejemplos/` con python ingestion.py, explica en dos frases qué comprueba load_corpus para no indexarlo y qué produciría sin esa comprobación con ese mismo archivo: un fragmento hecho de [page=1] [page=2], sin una palabra del documento y sin ninguna excepción.**

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import subprocess
import csv
import pandas as pd
from dotenv import load_dotenv

In [ ]:
# Raíz del proyecto: un nivel arriba de notebooks/
raiz = Path.cwd().parent

sys.path.insert(0, str(raiz))

print("Raíz del proyecto:", raiz)

In [ ]:
ejemplos = raiz / "ejemplos"
pdf_escaneado = ejemplos / "instructivo_escaneado.pdf"

In [ ]:
print("Directorio del proyecto:", raiz)
print("Carpeta ejemplos:", ejemplos)
print("PDF escaneado:", pdf_escaneado)

print("\nComprobaciones:")
print("ingestion.py:", (raiz / "ingestion.py").is_file())
print("rag_pipeline.py:", (raiz / "rag_pipeline.py").is_file())
print("carpeta ejemplos:", ejemplos.is_dir())
print("PDF escaneado:", pdf_escaneado.is_file())

Se procede a ejecutar el archivo `rag_pipeline.py`.

In [ ]:
os.environ["EMBEDDING_BACKEND"] = "local"
os.environ["QDRANT_URL"] = ":memory:"
os.environ["CORPUS_DIR"] = "ejemplos"

print("EMBEDDING_BACKEND =", os.environ["EMBEDDING_BACKEND"])
print("QDRANT_URL        =", os.environ["QDRANT_URL"])
print("CORPUS_DIR        =", os.environ["CORPUS_DIR"])

In [ ]:
resultado = subprocess.run(
    [sys.executable, str(raiz / "rag_pipeline.py")],
    capture_output=True,
    text=True
)

print(resultado.stdout)

if resultado.stderr:
    print("STDERR:")
    print(resultado.stderr)

print("Código de retorno:", resultado.returncode)

In [ ]:
resultado.stdout

In [ ]:
# Crear carpeta de resultados
carpeta_resultados =  raiz / "resultados" / "parte0"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

# Guardar stdout en CSV
archivo_csv = carpeta_resultados / "salida_0a.csv"

with open(archivo_csv, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["salida_stdout"])
    writer.writerow([resultado.stdout])

print(f"\nSalida guardada en: {archivo_csv}")

La salida de la ingesta fue almacenada dentro de la carpeta `resultados` dentro del presente repositorio. Mientras que las dos frases concluyentes que comprueban la funcionalidad de `load_corpus` y la consistencia de su salida, se describen al final del script `ingestion.py`.

## 0.b El fragmento que se corta a 128 

**Con el MiniLM de la Parte 0 (EMBEDDING_BACKEND=local, tope de 128 tokens), ingesta ejemplos/ pidiendo chunk_tokens=900 y pega el aviso. Después toma un texto de unas 900 palabras, cuenta sus tokens con el tokenizador del modelo, y compara el vector del texto entero con el del texto recortado a 
max_seq_length tokens: el coseno es ≈ 1. Todo lo que no cupo nunca llegó al índice. Es el ejercicio 5.1 del lunes, sobre el laboratorio.**

In [ ]:
from pathlib import Path
from rag_pipeline import RagPipeline
import os
import numpy as np

In [ ]:


os.environ["EMBEDDING_BACKEND"] = "local"
os.environ["QDRANT_URL"] = ":memory:"
os.environ["CORPUS_DIR"] = "ejemplos"

print("EMBEDDING_BACKEND =", os.environ["EMBEDDING_BACKEND"])
print("QDRANT_URL        =", os.environ["QDRANT_URL"])

In [ ]:


pipeline = RagPipeline()

chunks_900 = pipeline.ingest(
    Path("ejemplos"),
    chunk_tokens=900
)

print(f"\nFragmentos obtenidos: {len(chunks_900)}")
print(f"Tope del modelo: {pipeline.max_seq_tokens} tokens")

In [ ]:
faqs = [
    "Para resetear tu contraseña entra a Configuración > Seguridad y pulsa 'Olvidé mi clave'.",
    "Los reembolsos se procesan en 5 a 7 días hábiles tras aprobar la solicitud.",
    "Puedes exportar tus datos en formato CSV desde el panel de administración.",
    "La factura electrónica se emite automáticamente al confirmar el pago.",
    "Para cancelar tu suscripción ve a Cuenta > Plan y selecciona 'Cancelar'.",
    "El soporte técnico atiende de lunes a viernes de 9h a 18h por chat.",
    "La API permite hasta 100 solicitudes por minuto en el plan básico.",
    "Aceptamos tarjetas de crédito, débito y transferencias bancarias.",
]

# Texto de unas 900 palabras, como en el Ejercicio 5.1 del lunes
texto_largo = (" ".join(faqs) + " ") * 8

print("Palabras del texto:", len(texto_largo.split()))

In [ ]:
### se obtiene el tokenizador y el tope del modelo

modelo = pipeline.encoder._modelo

tokenizador = modelo.tokenizer
tope = modelo.max_seq_length

print("Modelo:", modelo)
print("max_seq_length:", tope)

In [ ]:
### se cuentan los tokens y se construye el recorte

ids = tokenizador(
    texto_largo,
    add_special_tokens=True,
    truncation=False
)["input_ids"]

# se conservan los tokens que caben dentro del límite del modelo.
recorte = tokenizador.decode(ids[1:tope - 1])

print("Tokens del texto:", len(ids))
print("Tope del modelo:", tope)
print("Tokens del recorte:", len(
    tokenizador(
        recorte,
        add_special_tokens=True,
        truncation=False
    )["input_ids"]
))

In [ ]:
### se comparan los embeddings


v_todo, v_recorte = pipeline.encoder.encode(
    [texto_largo, recorte]
)

coseno = float(v_todo @ v_recorte)

print(f"Tokens del texto: {len(ids)}")
print(f"Max_seq_length del modelo: {tope}")
print(f"Coseno(texto entero, recortado a {tope} tokens) = {coseno:.4f}")


Se guardan los resultados del ejercicio dentro de la carpeta `resultados/parte0` incluidos los tokens y el coseno.

In [ ]:
carpeta_resultados = raiz / "resultados" / "parte0"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

archivo = carpeta_resultados / "salida_0b.csv"

with open(archivo, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)

    writer.writerow(["elemento", "valor"])
    writer.writerow(["tokens_del_texto", len(ids)])
    writer.writerow(["max_seq_length", tope])
    writer.writerow(["tokens_del_recorte", len(
        tokenizador(
            recorte,
            add_special_tokens=True,
            truncation=False
        )["input_ids"]
    )])
    writer.writerow(["coseno", coseno])
    writer.writerow(["chunk_tokens_prueba", 900])
    writer.writerow(["chunk_tokens_parte1", 512])

print(f"Resultados guardados en: {archivo}")

In [ ]:
N = len(ids)
N

In [ ]:
coseno

### Conclusión


El texto de prueba contiene aproximadamente 900 palabras y el tokenizador
del MiniLM produjo [1106] tokens, mientras que el `max_seq_length` del modelo
es de 128 tokens. El coseno entre el embedding del texto completo y el
embedding del texto recortado a 128 tokens fue [COSENO=1], mostrando que el
modelo representa únicamente la parte de la entrada que cabe dentro de su
tope; el contenido restante no llega al embedding ni, por tanto, al índice.

Para la Parte 1 utilizaré `chunk_tokens=512`. Aunque bge-m3 permite hasta
8192 tokens, que un fragmento quepa en el límite del modelo no significa
que sea un buen fragmento para recuperación: un fragmento demasiado grande
puede mezclar demasiada información y reducir la precisión con la que un
vecino recuperado responde a una pregunta concreta.

## 0.c El índice que no se queja

**Sobre 
ejemplos/ indexado, pregunta algo cuya respuesta no está (
"¿cuál es la política de mascotas?"
) y
pega los tres vecinos con sus puntajes. Tienen aspecto sano. Después pregunta algo cuya respuesta sí está en el
PDF escaneado que la 0.a no indexó (
"¿qué exige el instructivo de prácticas de campo sobre el seguro
de accidentes?"
) y pega los tres vecinos. También tienen aspecto sano.**

In [ ]:
import os

os.environ["EMBEDDING_BACKEND"] = "local"
os.environ["QDRANT_URL"] = ":memory:"
os.environ["CORPUS_DIR"] = "ejemplos"

print("EMBEDDING_BACKEND =", os.environ["EMBEDDING_BACKEND"])
print("QDRANT_URL =", os.environ["QDRANT_URL"])
print("CORPUS_DIR =", os.environ["CORPUS_DIR"])

In [ ]:
from pathlib import Path
from rag_pipeline import RagPipeline

pipeline = RagPipeline()

chunks = pipeline.ingest(
   raiz / "ejemplos"
)

print(f"Fragmentos obtenidos: {len(chunks)}")
print(f"Tope del modelo: {pipeline.max_seq_tokens} tokens")

In [ ]:
## indexando a quadrant en memoria
pipeline.index(chunks)

print("Índice creado correctamente.")
print("Colección:", pipeline.collection)

In [ ]:
pregunta_1 = "¿Cuál es la política de mascotas?"

resultados_1 = pipeline.retrieve(
    pregunta_1,
    top_k=3
)

print("PREGUNTA:")
print(pregunta_1)
print("\nTOP 3 VECINOS:\n")

for i, resultado in enumerate(resultados_1, start=1):
    print(f"--- Vecino {i} ---")
    print(f"Score:    {resultado['score']:.4f}")
    print(f"Documento: {resultado['document']}")
    print(f"Chunk ID:  {resultado['chunk_id']}")
    print(f"Texto:    {resultado['text']}")
    print()

In [ ]:
# Crear una fila por cada vecino
datos_1 = []

for i, resultado in enumerate(resultados_1, start=1):
    datos_1.append({
        "pregunta": pregunta_1,
        "rank": i,
        "score": resultado["score"],
        "documento": resultado["document"],
        "chunk_id": resultado["chunk_id"],
        "texto": resultado["text"]
    })

# Crear DataFrame
df_0c_1 = pd.DataFrame(datos_1)

In [ ]:
pregunta_2 = (
    "¿Qué exige el instructivo de prácticas de campo "
    "sobre el seguro de accidentes?"
)

resultados_2 = pipeline.retrieve(
    pregunta_2,
    top_k=3
)

print("PREGUNTA:")
print(pregunta_2)
print("\nTOP 3 VECINOS:\n")

for i, resultado in enumerate(resultados_2, start=1):
    print(f"--- Vecino {i} ---")
    print(f"Score:    {resultado['score']:.4f}")
    print(f"Documento: {resultado['document']}")
    print(f"Chunk ID:  {resultado['chunk_id']}")
    print(f"Texto:    {resultado['text']}")
    print()

In [ ]:
# Crear una fila por cada vecino
datos_2 = []

for i, resultado in enumerate(resultados_2, start=1):
    datos_2.append({
        "pregunta": pregunta_2,
        "rank": i,
        "score": resultado["score"],
        "documento": resultado["document"],
        "chunk_id": resultado["chunk_id"],
        "texto": resultado["text"]
    })

# Crear DataFrame
df_0c_2 = pd.DataFrame(datos_2)


In [ ]:
df_0c_2

In [ ]:
df_0c = pd.concat(
    [df_0c_1, df_0c_2],
    ignore_index=True
)

display(df_0c)

In [ ]:
carpeta_resultados = raiz / "resultados" / "parte0"
carpeta_resultados.mkdir(parents=True, exist_ok=True)

archivo = carpeta_resultados / "salida_0c.csv"

df_0c.to_csv(
    archivo,
    index=False,
    encoding="utf-8-sig"
)

print(f"Resultados guardados en: {archivo}")

### Conclusiones

1. En esta parte no se calcula Hit Rate; los valores mostrados son los scores de los tres vecinos recuperados. El Hit Rate mide si el sistema encuentra al menos un fragmento relevante entre los primeros resultados. Sin embargo, no permite saber si una pregunta tiene realmente respuesta en el corpus.

2. Como se puede observar en los resultados de ambas preguntas, en ninguno de los tres vecinos se respondió correctamente (ver columna `texto` de la tabla). Además, en todas las salidas, los valores de los scores son demasiado bajos, y esto se debe porque para la primera pregunta ¿Cuál es la política de mascotas?, la respuesta no está en el corpus, por lo que Qdrant trata de buscar los fragmentos que considera más parecidos. Mientras que para la segunda pregunta ¿Qué exige el instructivo de prácticas de campo sobre el seguro de accidentes? , el pdf no tiene capa de texto al ser un documento escaneado, por lo que para el índice la información no existe.